<a href="https://colab.research.google.com/github/namsarena/yelp-complaint-tagger/blob/main/notebooks/yelp_complaint_tagger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1 – Setup.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -U datasets transformers accelerate evaluate scikit-learn

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import whoami
whoami()["name"]

In [ ]:
import datasets, transformers, pyarrow
print("datasets", datasets.__version__, "| transformers", transformers.__version__, "| pyarrow", pyarrow.__version__)

## Step 2 – Load and explore data.

In [ ]:
from datasets import load_dataset

ds = load_dataset("Yelp/yelp_review_full", split="train")      # 650,000 reviews
neg = ds.filter(lambda x: x["label"] <= 1)                      # label 0 = 1 star, 1 = 2 stars
neg = neg.filter(lambda x: 200 <= len(x["text"]) <= 1500)       # skip one-liners and essays
sample = neg.shuffle(seed=42).select(range(2500))               # same 2,500 every run

df = sample.to_pandas()
df["stars"] = df["label"] + 1
print(f"{len(neg):,} bad reviews available, working with {len(df):,}")
df["stars"].value_counts()

In [ ]:
df["length"] = df["text"].str.len()
print(df["length"].describe().round())

In [ ]:
for i, t in enumerate(df["text"].head(10)):
    print(f"#{i}  ({df['stars'][i]}★)\n{t}\n" + "-" * 80)

In [ ]:
# My hand tags (answer key) — labels v1.1, after the pilot review
my_tags = {
    0: {"labels": ["food_quality"], "note": "bakery; flavor + dry"},
    1: {"labels": ["wait_time", "price_value"], "note": "bar counts; forgotten = wait_time"},
    2: {"labels": ["price_value"], "note": "reread with v1.1 rules"},
    3: {"labels": ["food_quality"], "note": "cute ambiance is praise"},
    4: {"labels": ["NOT_RESTAURANT"], "note": ""},
    5: {"labels": ["NOT_RESTAURANT"], "note": "hotel"},
    6: {"labels": ["order_accuracy", "food_quality"], "note": "missing boba, sold out, over-blended"},
    7: {"labels": ["NOT_RESTAURANT"], "note": ""},
    8: {"labels": ["food_quality", "ambience", "price_value"], "note": "singer annoying; not worth it"},
    9: {"labels": ["NOT_RESTAURANT"], "note": ""},
}
print("Answer key v1.1 loaded:", my_tags[0]["labels"])

In [ ]:
import pandas as pd

ALLOWED = {"wait_time", "staff_attitude", "food_quality", "order_accuracy", "price_value",
           "hygiene_safety", "ambience", "billing_policy", "NOT_RESTAURANT", "???"}

# 1) catch typos
for i, t in my_tags.items():
    bad = [l for l in t["labels"] if l not in ALLOWED]
    if bad:  print(f"#{i}: unknown label {bad}, check spelling")
    if not t["labels"]: print(f"#{i}: no labels yet")

# 2) summary
hand = pd.DataFrame([{"id": i, "text": df["text"][i], "labels": ", ".join(t["labels"]), "note": t["note"]}
                     for i, t in my_tags.items()])
counts = hand["labels"].str.split(", ").explode().value_counts()
print("\nLabel counts:\n", counts)
print("\nReviews marked ??? :", (hand["labels"].str.contains(r"\?\?\?")).sum())

# 3) save as a file (start of your gold test set)
hand.to_csv("hand_tags_first10.csv", index=False)
print("\nSaved hand_tags_first10.csv")

 ## Step 3 – Label with a free LLM

In [ ]:
# 3.2 Save work to a PRIVATE Hugging Face dataset (no Google Drive access needed)
from huggingface_hub import HfApi
import os

api = HfApi()
DATA_REPO = "nrr26/yelp-complaint-labels"
api.create_repo(DATA_REPO, repo_type="dataset", private=True, exist_ok=True)

OUT = "/content/out"                      # local working folder
os.makedirs(OUT, exist_ok=True)

def save_to_hub(local_file, message):
    api.upload_file(path_or_fileobj=local_file, path_in_repo=os.path.basename(local_file),
                    repo_id=DATA_REPO, repo_type="dataset", commit_message=message)
    print("Uploaded", os.path.basename(local_file), "to", DATA_REPO)

hand.to_csv(f"{OUT}/hand_tags_first10.csv", index=False)
save_to_hub(f"{OUT}/hand_tags_first10.csv", "Hand tags for first 10 reviews (labels v1.0)")

In [ ]:
# 3.3 About 30% of reviews are not restaurants, so take 3,500 to keep about 2,400
sample = neg.shuffle(seed=42).select(range(3500))
df = sample.to_pandas()
df["stars"] = df["label"] + 1
assert df["text"][0] == hand["text"][0], "order changed!"
print(len(df), "reviews; first 10 are still your hand-tagged ones")

In [ ]:
# 3.4 Free open model (Apache 2.0) that runs on the Colab T4 GPU load the free open model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen3-4B-Instruct-2507"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"                       # required for batched generation
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")
print("Loaded", MODEL)

In [ ]:
# 3.5 Prompt + helpers (Write the labelling instructions)
import json, re

LABELS = {
    "wait_time": "slow seating, slow food, ignored at the table",
    "staff_attitude": "rude, dismissive or unprofessional staff",
    "food_quality": "taste, freshness, cooking, cold food",
    "order_accuracy": "wrong or missing items",
    "price_value": "overpriced, tiny portions",
    "hygiene_safety": "dirty, hair or bugs in food, got sick",
    "ambience": "noise, seating, temperature, decor",
    "billing_policy": "overcharged, refunds, reservation rules, manager handling",
}
DEFS = "\n".join(f"- {k}: {v}" for k, v in LABELS.items())

def build_prompt(text):
    user = f"""Read this 1-2 star Yelp review.
1. Is it about a restaurant, bar, cafe or other food place? Hotels, salons, shops and doctors are NOT.
2. If yes, list every complaint category the reviewer explicitly mentions. Use only these labels:
{DEFS}

Reply with JSON only: {{"is_restaurant": true or false, "labels": [...]}}

Review:
\"\"\"{text}\"\"\""""
    msgs = [{"role": "system", "content": "You label customer reviews. Reply with JSON only."},
            {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse(raw):
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        d = json.loads(m.group(0))
        return bool(d.get("is_restaurant")), [l for l in d.get("labels", []) if l in LABELS]
    except Exception:
        return None, []                         # unreadable answer; we count these

@torch.no_grad()
def label_batch(texts):
    enc = tok([build_prompt(t) for t in texts], return_tensors="pt", padding=True).to("cuda")
    out = llm.generate(**enc, max_new_tokens=60, do_sample=False)
    raw = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [parse(r) for r in raw], raw

In [ ]:
# 3.6 Pilot: compare the LLM with my 10 hand tags
res, raw = label_batch(df["text"][:10].tolist())

rows, tp, fp, fn = [], 0, 0, 0
for i, ((is_rest, labs), r) in enumerate(zip(res, raw)):
    mine = set(my_tags[i]["labels"])
    llm_set = {"NOT_RESTAURANT"} if is_rest is False else set(labs)
    if is_rest is None: llm_set = {"UNPARSEABLE"}
    tp += len(mine & llm_set); fp += len(llm_set - mine); fn += len(mine - llm_set)
    rows.append({"#": i, "mine": ", ".join(sorted(mine)), "llm": ", ".join(sorted(llm_set)),
                 "match": "✅" if mine == llm_set else "❌"})

print(pd.DataFrame(rows).to_string(index=False))
print(f"\nExact matches: {sum(r['match']=='✅' for r in rows)}/10")
print(f"Precision {tp/max(tp+fp,1):.0%}  (of the LLM's labels, how many you agree with)")
print(f"Recall    {tp/max(tp+fn,1):.0%}  (of your labels, how many the LLM found)")

In [ ]:
# 3.6b Score the two decisions separately
rest_ok, tp, fp, fn = 0, 0, 0, 0
for i, (is_rest, labs) in enumerate(res):
    mine = set(my_tags[i]["labels"])
    mine_rest = "NOT_RESTAURANT" not in mine
    rest_ok += (mine_rest == bool(is_rest))
    if mine_rest and is_rest:                    # both say restaurant: compare complaints
        llm_set = set(labs)
        tp += len(mine & llm_set); fp += len(llm_set - mine); fn += len(mine - llm_set)

print(f"Restaurant or not: {rest_ok}/10 agree")
print(f"Complaint labels: precision {tp/max(tp+fp,1):.0%}, recall {tp/max(tp+fn,1):.0%}")

In [ ]:
for i in [0, 1, 3, 6, 8]:
    print(f"#{i}  mine: {my_tags[i]['labels']}  |  llm: {res[i]}\n{df['text'][i]}\n" + "-" * 80)
# Read each review and decide for each one: were you right, or was the LLM? Be honest; you might have missed something.

In [ ]:
# Which versions are actually in memory?
print("my_tags #0  :", my_tags[0]["labels"])                     # new = ['food_quality']
print("prompt v2?  :", "Examples:" in build_prompt("test"))      # new = True
print("LLM said #3 :", res[3])                                   # v2 should drop 'ambience'

In [ ]:
# 3.2b Answer key v1.1 -> rebuild table -> upload (all in one cell)
my_tags = {
    0: {"labels": ["food_quality"], "note": "bakery; flavor + dry"},
    1: {"labels": ["wait_time", "price_value"], "note": "bar counts; forgotten = wait_time"},
    2: {"labels": ["price_value"], "note": "reread with v1.1 rules"},
    3: {"labels": ["food_quality"], "note": "cute ambiance is praise"},
    4: {"labels": ["NOT_RESTAURANT"], "note": ""},
    5: {"labels": ["NOT_RESTAURANT"], "note": "hotel"},
    6: {"labels": ["order_accuracy", "food_quality"], "note": "missing boba, sold out, over-blended"},
    7: {"labels": ["NOT_RESTAURANT"], "note": ""},
    8: {"labels": ["food_quality", "ambience", "price_value"], "note": "singer annoying; not worth it"},
    9: {"labels": ["NOT_RESTAURANT"], "note": ""},
}

hand = pd.DataFrame([{"id": i, "text": df["text"][i], "labels": ", ".join(t["labels"]), "note": t["note"]}
                     for i, t in my_tags.items()])
hand.to_csv(f"{OUT}/hand_tags_first10.csv", index=False)
save_to_hub(f"{OUT}/hand_tags_first10.csv", "Answer key v1.1 after pilot review")

print(hand[["id", "labels"]].to_string(index=False))

In [ ]:
# 3.5b + 3.6 + 3.6b  Prompt v2 -> pilot run -> scores (all in one cell)

# ---- Prompt v2: label definitions v1.1, rules, few-shot examples ----
LABELS = {
    "wait_time": "slow seating, slow food, slow bill, forgotten or ignored by staff",
    "staff_attitude": "rude, dismissive or disrespectful staff (NOT slowness, NOT mistakes)",
    "food_quality": "taste, freshness, dryness, cooking, temperature, drinks made badly",
    "order_accuracy": "wrong, missing or sold-out items",
    "price_value": "overpriced, tiny portions, 'not worth it'",
    "hygiene_safety": "dirty, hair or bugs in food, got sick",
    "ambience": "complaints about noise, music, seating, temperature, decor",
    "billing_policy": "overcharged or wrong bill, refunds, reservation rules, manager handling",
}
DEFS = "\n".join(f"- {k}: {v}" for k, v in LABELS.items())

RULES = """Rules:
- Restaurants include bars, cafes, bakeries, dessert and boba shops, and food trucks. Hotels, salons, shops, doctors and gyms are NOT.
- Tag complaints only. Praise or neutral description gets no label ("cute decor" is not an ambience complaint).
- Tag only what the reviewer states. Do not guess.
- If is_restaurant is false, labels must be []."""

EXAMPLES = """Examples:
Review: "Cute patio and friendly servers, but my steak was raw and $40 for it is a joke."
{"is_restaurant": true, "labels": ["food_quality", "price_value"]}
Review: "Waited 30 minutes for a table, then the bartender rolled his eyes when we asked for water."
{"is_restaurant": true, "labels": ["wait_time", "staff_attitude"]}
Review: "Ordered a chicken burrito, got beef. Half the menu was sold out."
{"is_restaurant": true, "labels": ["order_accuracy"]}
Review: "The DJ was so loud we had to shout, and they charged us twice."
{"is_restaurant": true, "labels": ["ambience", "billing_policy"]}
Review: "Front desk lost our room booking and the pool was dirty."
{"is_restaurant": false, "labels": []}"""

def build_prompt(text):
    user = f"""Read this 1-2 star Yelp review.
1. Is it about a restaurant or other food or drink place?
2. If yes, list every complaint category the reviewer states. Use only these labels:
{DEFS}

{RULES}

{EXAMPLES}

Reply with JSON only: {{"is_restaurant": true or false, "labels": [...]}}

Review:
\"\"\"{text}\"\"\""""
    msgs = [{"role": "system", "content": "You label customer reviews. Reply with JSON only."},
            {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print("Prompt v2 ready. Running pilot on 10 reviews...\n")

# ---- Pilot: ask the LLM to label the 10 reviews ----
res, raw = label_batch(df["text"][:10].tolist())

# ---- Table: my answer key vs the LLM ----
rows = []
for i, (is_rest, labs) in enumerate(res):
    mine = set(my_tags[i]["labels"])
    llm_set = {"NOT_RESTAURANT"} if is_rest is False else ({"UNPARSEABLE"} if is_rest is None else set(labs))
    rows.append({"#": i, "mine": ", ".join(sorted(mine)), "llm": ", ".join(sorted(llm_set)),
                 "match": "✅" if mine == llm_set else "❌"})
print(pd.DataFrame(rows).to_string(index=False))

# ---- Scores: restaurant decision and complaint labels, separately ----
rest_ok, tp, fp, fn = 0, 0, 0, 0
for i, (is_rest, labs) in enumerate(res):
    mine = set(my_tags[i]["labels"])
    mine_rest = "NOT_RESTAURANT" not in mine
    rest_ok += (mine_rest == bool(is_rest))
    if mine_rest and is_rest:
        llm_set = set(labs)
        tp += len(mine & llm_set); fp += len(llm_set - mine); fn += len(mine - llm_set)

print(f"\nExact matches: {sum(r['match']=='✅' for r in rows)}/10")
print(f"Restaurant or not: {rest_ok}/10 agree")
print(f"Complaint labels: precision {tp/max(tp+fp,1):.0%}, recall {tp/max(tp+fn,1):.0%}")

In [ ]:
# 3.7 Full run: label all 3,500 reviews with prompt v2, saving to the Hub every 400 rows
import time

BATCH = 16                                   # reviews per GPU call; use 8 if you get "out of memory"
rows_full = globals().get("rows_full", [])   # resume support: keeps rows if you rerun this cell
start = len(rows_full)
t0 = time.time()
print(f"Starting at review {start} of {len(df)}")

for s in range(start, len(df), BATCH):
    texts = df["text"][s:s + BATCH].tolist()
    parsed, raws = label_batch(texts)
    for j, ((is_rest, labs), r) in enumerate(zip(parsed, raws)):
        rows_full.append({"id": s + j, "text": texts[j], "stars": int(df["stars"][s + j]),
                          "is_restaurant": is_rest, "labels": ", ".join(labs), "raw": r})
    done = len(rows_full)
    if done % 400 < BATCH or done == len(df):            # checkpoint about every 400 rows
        pd.DataFrame(rows_full).to_csv(f"{OUT}/llm_labels_v2.csv", index=False)
        save_to_hub(f"{OUT}/llm_labels_v2.csv", f"LLM labels (prompt v2): {done}/{len(df)}")
        rate = (done - start) / (time.time() - t0)
        print(f"  {done}/{len(df)} done, about {(len(df) - done) / rate / 60:.0f} min left")

print("✅ Finished:", len(rows_full), "reviews labelled")

In [ ]:
# 4.1 Clean the LLM labels and count them
lab = pd.DataFrame(rows_full)
print("Total labelled:", len(lab))
print("Unreadable LLM answers:", lab["is_restaurant"].isna().sum())
print("Not restaurants (dropped):", (lab["is_restaurant"] == False).sum())

lab = lab[lab["is_restaurant"] == True].copy()
lab["label_list"] = lab["labels"].fillna("").apply(lambda s: [l for l in s.split(", ") if l])
print("Restaurant reviews kept:", len(lab))
print("Kept but no complaint label:", (lab["label_list"].str.len() == 0).sum())
print("\nExamples per label:")
print(lab["label_list"].explode().value_counts())

In [ ]:
# 4.2 Split: train (85%) / test (15%) / gold (my hand-checked pilot reviews)
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

LABEL_NAMES = list(LABELS)                               # fixed order of the 8 labels

lab = lab[lab["id"] >= 10]                                # pilot reviews go to gold, not train
gold = pd.DataFrame([{"text": df["text"][i], "label_list": t["labels"]}
                     for i, t in my_tags.items() if "NOT_RESTAURANT" not in t["labels"]])
train_df, test_df = train_test_split(lab, test_size=0.15, random_state=42)

def to_dataset(d):
    return Dataset.from_dict({
        "text": d["text"].tolist(),
        "labels": [[1.0 if n in ls else 0.0 for n in LABEL_NAMES] for ls in d["label_list"]],
        "label_names": [", ".join(ls) for ls in d["label_list"]],
    })

data = DatasetDict(train=to_dataset(train_df), test=to_dataset(test_df), gold=to_dataset(gold))
data.push_to_hub(DATA_REPO, private=True, commit_message="Step 4: train/test/gold splits (labels v1.1, prompt v2)")
print(data)
print("Label order:", LABEL_NAMES)

In [ ]:
# 4.3 Unload the 8 GB labeller model; the GPU is needed for training next
import gc
if "llm" in locals() or "llm" in globals():
    del llm
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# 5.1 Baseline: TF-IDF + logistic regression (one classifier per label)
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report

ytr = np.array(data["train"]["labels"]).astype(int)
yte = np.array(data["test"]["labels"]).astype(int)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
Xtr = vec.fit_transform(data["train"]["text"])          # learn vocabulary from TRAIN only
Xte = vec.transform(data["test"]["text"])

base = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))
base.fit(Xtr, ytr)
pred = base.predict(Xte)

print(classification_report(yte, pred, target_names=LABEL_NAMES, zero_division=0))
baseline_scores = {"micro_f1": f1_score(yte, pred, average="micro"),
                   "macro_f1": f1_score(yte, pred, average="macro")}
print(f"BASELINE  micro-F1 {baseline_scores['micro_f1']:.2f}  |  macro-F1 {baseline_scores['macro_f1']:.2f}")

In [ ]:
# 5.2 Top words per label: does the model learn sensible signals?
words = np.array(vec.get_feature_names_out())
for k, name in enumerate(LABEL_NAMES):
    top = words[np.argsort(base.estimators_[k].coef_[0])[-8:][::-1]]
    print(f"{name:15s} {', '.join(top)}")

In [ ]:
# 6.1 Tokenize + carve a validation set out of train
from transformers import AutoTokenizer
from datasets import Sequence, Value

BASE = "distilbert-base-uncased"
btok = AutoTokenizer.from_pretrained(BASE)

def tokenize(batch):
    return btok(batch["text"], truncation=True, max_length=384)

tok_data = data.map(tokenize, batched=True, remove_columns=["text", "label_names"])
tok_data = tok_data.cast_column("labels", Sequence(Value("float32")))

split = tok_data["train"].train_test_split(test_size=0.1, seed=42)    # 1,800 train / 200 validation
train_set, val_set, test_set = split["train"], split["test"], tok_data["test"]
print(f"train {len(train_set)} | validation {len(val_set)} | test {len(test_set)}")

In [ ]:
# 6.2 Fine-tune DistilBERT for multi-label classification
import numpy as np
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=len(LABEL_NAMES), problem_type="multi_label_classification",
    id2label=dict(enumerate(LABEL_NAMES)), label2id={n: i for i, n in enumerate(LABEL_NAMES)})

def compute_metrics(p):
    probs = 1 / (1 + np.exp(-p.predictions))          # sigmoid: one yes/no probability per label
    preds = (probs > 0.5).astype(int)
    return {"micro_f1": f1_score(p.label_ids, preds, average="micro", zero_division=0),
            "macro_f1": f1_score(p.label_ids, preds, average="macro", zero_division=0)}

args = TrainingArguments(
    output_dir="/content/out/distilbert",
    num_train_epochs=4, learning_rate=3e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    fp16=True, logging_steps=25, report_to="none", seed=42)

trainer = Trainer(model=model, args=args, train_dataset=train_set, eval_dataset=val_set,
                  data_collator=DataCollatorWithPadding(btok), compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# 6.3 Final test: DistilBERT vs baseline, per label (test set used once)
logits = trainer.predict(test_set).predictions
bert_pred = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)

cmp = pd.DataFrame({"label": LABEL_NAMES,
                    "baseline_F1": f1_score(yte, pred, average=None, zero_division=0),
                    "distilbert_F1": f1_score(yte, bert_pred, average=None, zero_division=0)}).round(2)
cmp["change"] = (cmp["distilbert_F1"] - cmp["baseline_F1"]).round(2)
print(cmp.to_string(index=False))
for avg in ["micro", "macro"]:
    b, d = f1_score(yte, pred, average=avg), f1_score(yte, bert_pred, average=avg)
    print(f"{avg}-F1: baseline {b:.2f} -> DistilBERT {d:.2f}  ({d - b:+.2f})")

In [ ]:
# 6.4 Retrain: weighted loss for rare labels, 8 epochs, lr 5e-5
import gc, torch
if "trainer" in locals() or "trainer" in globals():
    del trainer
if "model" in locals() or "model" in globals():
    del model
gc.collect(); torch.cuda.empty_cache()      # free the first model

y_train = np.array(train_set["labels"])
pos = y_train.sum(0); neg = len(y_train) - pos
pos_weight = torch.tensor(np.clip(neg / pos, 1, 10), dtype=torch.float32)
print("Extra weight per label:", dict(zip(LABEL_NAMES, pos_weight.numpy().round(1))))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            outputs.logits, labels, pos_weight=pos_weight.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=len(LABEL_NAMES), problem_type="multi_label_classification",
    id2label=dict(enumerate(LABEL_NAMES)), label2id={n: i for i, n in enumerate(LABEL_NAMES)})

args = TrainingArguments(
    output_dir="/content/out/distilbert_v2",
    num_train_epochs=8, learning_rate=5e-5, warmup_steps=50, weight_decay=0.01,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    fp16=True, logging_steps=25, report_to="none", seed=42)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_set, eval_dataset=val_set,
                          data_collator=DataCollatorWithPadding(btok), compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# 6.5 Best threshold per label (chosen on VALIDATION), then the final TEST comparison
sigmoid = lambda x: 1 / (1 + np.exp(-x))
yval = np.array(val_set["labels"]).astype(int)
val_probs = sigmoid(trainer.predict(val_set).predictions)

grid = np.arange(0.10, 0.91, 0.05)
thresholds = np.array([grid[np.argmax([f1_score(yval[:, k], (val_probs[:, k] > t).astype(int), zero_division=0)
                                       for t in grid])] for k in range(len(LABEL_NAMES))])

test_probs = sigmoid(trainer.predict(test_set).predictions)
bert_pred = (test_probs > thresholds).astype(int)

cmp = pd.DataFrame({"label": LABEL_NAMES, "threshold": thresholds.round(2),
                    "baseline_F1": f1_score(yte, pred, average=None, zero_division=0),
                    "distilbert_F1": f1_score(yte, bert_pred, average=None, zero_division=0)}).round(2)
cmp["change"] = (cmp["distilbert_F1"] - cmp["baseline_F1"]).round(2)
print(cmp.to_string(index=False))
for avg in ["micro", "macro"]:
    b, d = f1_score(yte, pred, average=avg), f1_score(yte, bert_pred, average=avg)
    print(f"{avg}-F1: baseline {b:.2f} -> DistilBERT {d:.2f}  ({d - b:+.2f})")

In [ ]:
# 7.1 Gold check: the model vs my hand-checked answers
trainer.model.eval()
gold_probs = sigmoid(trainer.predict(tok_data["gold"]).predictions)
gold_pred = gold_probs > thresholds
for i in range(len(data["gold"])):
    model_labels = [LABEL_NAMES[k] for k in range(len(LABEL_NAMES)) if gold_pred[i, k]]
    print(f"mine : {data['gold']['label_names'][i]}\nmodel: {', '.join(model_labels)}\n")

In [ ]:
# 7.2 Tag any text: probability per label, kept if above that label's threshold
def tag(texts):
    enc = btok(texts, truncation=True, max_length=384, padding=True, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        probs = sigmoid(trainer.model(**enc).logits.float().cpu().numpy())
    return [{LABEL_NAMES[k]: round(float(p[k]), 2) for k in range(len(LABEL_NAMES)) if p[k] > thresholds[k]} for p in probs]

tests = [
    "We waited an hour for cold pasta and the waiter was rude when we complained.",
    "They forgot half our order and charged us twice. Manager refused a refund.",
    "The staff was not rude at all, but the music was way too loud.",
    "Amazing tacos, friendly staff, great prices! Will be back.",
    "Food was fine but we sat for 40 minutes before anyone took our order.",
    "Good ambiance, but food smelled and quality doesn't worth the price for buffet.",
    "Tea was empty, most food was not ready and no dessert, except dosa everything was cold.",
    "Dosa delivered was soggy.",
    "I waited to get food delivered and then at billing she did not offer the 10% discount promised for a $100 bill.",
    "Most of the food items were not tasty and I had to file a dispute.",
]
for t, labels in zip(tests, tag(tests)):
    print(f"{labels}\n  <- {t}\n")

In [ ]:
# 7.3 Save model + tokenizer + per-label thresholds (stored in config.json)
FINAL = "/content/out/final_model"
trainer.model.config.label_thresholds = {n: float(t) for n, t in zip(LABEL_NAMES, thresholds)}
trainer.save_model(FINAL)
btok.save_pretrained(FINAL)
print(sorted(os.listdir(FINAL)))

The card pulls your real numbers from the notebook, so there's no copying by hand and no typos.


In [ ]:
# 8.1 Model card (README.md) built from this notebook's real numbers
MODEL_REPO = "nrr26/yelp-complaint-tagger"
support = yte.sum(0)
rows_md = "\n".join(f"| `{r.label}` | {r.threshold:.2f} | {r.baseline_F1:.2f} | **{r.distilbert_F1:.2f}** | {s} |"
                    for r, s in zip(cmp.itertuples(), support))
nums = {"__MICRO_B__": f1_score(yte, pred, average="micro"), "__MICRO_D__": f1_score(yte, bert_pred, average="micro"),
        "__MACRO_B__": f1_score(yte, pred, average="macro"), "__MACRO_D__": f1_score(yte, bert_pred, average="macro")}

card = """---
license: cc-by-nc-4.0
language: en
library_name: transformers
pipeline_tag: text-classification
base_model: distilbert/distilbert-base-uncased
datasets:
- Yelp/yelp_review_full
tags:
- multi-label-classification
- reviews
- restaurants
- yelp
---

# Yelp complaint tagger

Tags **why** a restaurant got a bad review. Give it a 1-2 star review and it returns one or more of 8 complaint types:
`wait_time`, `staff_attitude`, `food_quality`, `order_accuracy`, `price_value`, `hygiene_safety`, `ambience`, `billing_policy`.

A small (66M parameters) DistilBERT model, fine-tuned on about 2,000 Yelp reviews labelled by an open LLM. Built as a learning project.

## Quick start

Each label has its own threshold, stored in `config.label_thresholds`. Use them; plain 0.5 gives worse results.

~~~python
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo = "nrr26/yelp-complaint-tagger"
tok = AutoTokenizer.from_pretrained(repo)
model = AutoModelForSequenceClassification.from_pretrained(repo).eval()
th = model.config.label_thresholds

text = "We waited 45 minutes and the burger was cold."
with torch.no_grad():
    probs = torch.sigmoid(model(**tok(text, return_tensors="pt", truncation=True)).logits)[0]
print({label: round(float(p), 2) for label, p in zip(model.config.id2label.values(), probs) if p > th[label]})
~~~

## Intended use

- **Use it on reviews already known to be negative** (1-2 stars) about restaurants, bars, cafes and similar places.
- Good for spotting patterns across many reviews, such as "30% of our complaints are about wait time".
- **Not for:** positive or mixed reviews (it was never trained on them and will invent complaints), non-restaurant businesses, languages other than English, or decisions about individual staff members.

## Labels and results

Test set: 354 held-out reviews labelled by the same LLM. F1 is between 0 and 1; higher is better. The baseline is TF-IDF + logistic regression.

| Label | Threshold | Baseline F1 | This model F1 | Test examples |
|---|---|---|---|---|
__ROWS__

| Average | Baseline | This model |
|---|---|---|
| micro-F1 | __MICRO_B__ | **__MICRO_D__** |
| macro-F1 | __MACRO_B__ | **__MACRO_D__** |

## How it was built

1. **Data:** 3,500 random 1-2 star reviews (200-1,500 characters) from `Yelp/yelp_review_full`, seed 42.
2. **Labels (silver):** `Qwen/Qwen3-4B-Instruct-2507` labelled each review with a written guide (8 labels plus rules) and 5 invented few-shot examples. It dropped 1,138 non-restaurant reviews, leaving 2,361.
   - Checked against 10 hand-labelled reviews: 70% precision and 70% recall per label, 9/10 on restaurant-or-not.
   - Spot check of 20 fresh reviews: about 18/20 judged correct.
3. **Training:** DistilBERT, 8 epochs, learning rate 5e-5, weighted loss (rare labels up to 10x), best epoch chosen by v7).
   - Split: 1,800 train / 200 validation / 354 test.
   - Per-label thresholds tuned on validation.

## Limitations

- **It copies its teacher.** The training and test labels come from an LLM, so the scores above measure agreement with tOn 6 hand-labelled reviews it matched exactly on 2 and repeated the LLM's habits: tagging praised ambience, reading slowor sloppy service as rude, and under-tagging `price_value`.
- **Negation:** "the staff was not rude" can still be tagged `staff_attitude`.
- **Positive reviews:** it assumes every text is a complaint.
- **`staff_attitude` over-fires** with weak scores (about 0.3); its threshold was tuned on only 200 validation reviews.
- **Secondary complaints** (sold-out items, "had to file a dispute") are sometimes missed.
- **Rare labels** (`billing_policy`, `order_accuracy`) have only 24-39 test examples, so their scores are noisy.
- **Process note:** the test set was looked at once before the weighted-loss retrain. All later choices used the validation set only.

## License and credits

- Model weights: **CC BY-NC 4.0** (non-commercial), because the training text comes from Yelp, whose dataset terms allow non-commercial use. The training reviews are not redistributed.
- Base model: DistilBERT (Apache 2.0). Labelling model: Qwen3-4B-Instruct (Apache 2.0).
"""
card = card.replace("__ROWS__", rows_md)
for k, v in nums.items():
    card = card.replace(k, f"{v:.2f}")
open(f"{FINAL}/README.md", "w").write(card)
print(card)

In [ ]:
# 8.2 Upload to the Hub as PRIVATE first
api.create_repo(MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=FINAL, repo_id=MODEL_REPO, ignore_patterns=["training_args.bin"],
                  commit_message="v1.0: DistilBERT complaint tagger (labels v1.1, prompt v2)")
print("Uploaded (private): https://huggingface.co/" + MODEL_REPO)

In [ ]:
# 8.3 Load from the Hub, not from memory, exactly as a user would
from transformers import AutoTokenizer, AutoModelForSequenceClassification
m = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO).eval()
t = AutoTokenizer.from_pretrained(MODEL_REPO)
th = m.config.label_thresholds
text = "We waited 45 minutes and the burger was cold."
with torch.no_grad():
    p = torch.sigmoid(m(**t(text, return_tensors="pt", truncation=True)).logits)[0]
print("Thresholds loaded:", th)
print({l: round(float(x), 2) for l, x in zip(m.config.id2label.values(), p) if x > th[l]})

In [ ]:
# 8.5 Tidy: round thresholds to 2 decimals and re-upload config.json only
m.config.label_thresholds = {k: round(v, 2) for k, v in th.items()}
m.config.save_pretrained(FINAL)
api.upload_file(path_or_fileobj=f"{FINAL}/config.json", path_in_repo="config.json",
                repo_id=MODEL_REPO, commit_message="Round thresholds to 2 decimals")
print(m.config.label_thresholds)